# Recipe Ablation of Ours (Full): why OrdinalCLIP (official) wins at low shot
## DeepDRiD-CFP · copy of `DeepDRiD_FewShot_DR_fixed.ipynb`

OrdinalCLIP (official) beats Ours (Full) at k = 1 and 2. Its recipe differs from ours in three ways.
This notebook adds them to Ours (Full) one, two and all three at a time (7 models) and
tests them at k = -1 (full data), 1, 2 and 4, with 3 seeds each.

| Code | Change on top of Ours (Full) | Overrides |
|------|------------------------------|-----------|
| **E** | 100 epochs, no warmup (ours: 50 epochs, 5 warmup) | `train.epochs=100`, `train.warmup_epochs=0` |
| **T** | Text-initialised prompt context, OrdinalCLIP sentence `"<ctx> <rank>."` (ours: random context) | `init_context='diabetic retinopathy grading: the severity of the retinopathy is'`, `full_stop=true` |
| **F** | Full fine-tuning of the RN50 image encoder (ours: LoRA only; LoRA stays attached) | `model.image_encoder.full_finetune=true` |

Everything else (LoRA, SMS, ranking loss, CLIP preprocessing, AdamW, splits, test set) is Ours (Full).

**Checkpoints.** Every variant has its own resolved config, so it gets its own folder
`results_shared/<config key>/` and never touches the Ours (Full) folder (`76d56d6ae883`).
The variant name is written to that folder's `names.txt`, and section 5d writes
`results_shared/recipe_ablation_index.csv` (name → folder). Tables and figures of this
notebook go to `results_recipe/`.


## 1 · Environment Setup

In [ ]:
import torch
import os, sys, json, warnings, logging
import subprocess
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from omegaconf import OmegaConf
from tqdm import tqdm
import sys
# import tqdm.std
# sys.modules['tqdm.notebook'] = tqdm.std  # Tricks PyTorch Lightning into using standard tqdm

warnings.filterwarnings('ignore')
logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)

# ── Locate project root (contains 'configs/' dir) ────────────────────────────
ROOT = Path.cwd()
while not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from scripts.train import load_config
from src.data.datamodule import FewShotDataModule
from src.models.runner import FewShotOrdinalRunner

# results_v3: per-eye labels, patient-grouped splits, and our model without the
# L2RCLIP parts (RankFormer, pairwise loss). Older runs in results/ and results_v2/
# are not reused.
RESULTS = ROOT / 'results_recipe'   # tables and figures of this notebook
# Runs are cached in results_shared/, keyed by their resolved config, and shared
# with the Direction 1-3 notebooks: a model trained in any of them is not retrained here.
from src.utils.experiment import find_shared_results, find_runs, cached_metrics
SHARED_RESULTS = find_shared_results(ROOT)
RESULTS.mkdir(exist_ok=True)
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Project root : {ROOT}')
print(f'Device       : {DEVICE}')
print(f'Results dir  : {RESULTS}')
print(f'Shared runs  : {SHARED_RESULTS}')
print(f'PyTorch      : {torch.__version__}')
print(f'PL           : {pl.__version__}')

## 1b · Auto-generate DeepDRiD Manifest

Converts raw DeepDRiD CSV files into the standardised `CFP_manifest.csv`.
Safe to re-run; silently skipped if manifest already exists.


In [ ]:
# Auto-generate the DeepDRiD manifest if it does not yet exist.
import subprocess as _sp
_manifest = ROOT / 'data/DeepDRiD' / 'CFP_manifest.csv'
if not _manifest.exists():
    print(f'Generating DeepDRiD manifest -> {_manifest}')
    r = _sp.run(
        [sys.executable, 'scripts/prepare_external_datasets.py', '--skip-aptos'],
        capture_output=True, text=True,
    )
    print(r.stdout)
    if r.returncode != 0:
        print('ERROR:', r.stderr)
else:
    print(f'Manifest OK: {_manifest} ({len(pd.read_csv(_manifest)):,} rows)')


## 2 · Experiment Grid

Edit these cells to control what gets trained / re-run.

# Configuration Options Reference

This table outlines all the available configuration overrides in the MMRDR framework. These keys can be passed to the `load_config(overrides=[...])` function to dynamically alter the model architecture, data loading, or training pipeline.

| Configuration Key | Type | Available Options / Examples | Description |
| :--- | :--- | :--- | :--- |
| **`runner`** | String | `"src.models.runner.FewShotOrdinalRunner"`, <br>`"src.models.runner.StandardRunner"` | The PyTorch Lightning module used to run the training/inference loop. |
| **`model.image_encoder.name`** | String | `"RN50"`, `"RN101"`, `"ViT-B/32"`, `"ViT-B/16"`, `"ViT-L/14"` | The backbone architecture for the vision encoder. |
| **`model.image_encoder.lora.enabled`** | Boolean | `true`, `false` | Toggles Low-Rank Adaptation (LoRA) on the image encoder for parameter-efficient fine-tuning. |
| **`model.image_encoder.lora.r`** | Integer | `2`, `4`, `8`, `16`, `32`, `64` | The rank (`r`) dimension used for LoRA projection matrices. |
| **`model.text_encoder.name`** | String | `"RN50"`, `"ViT-B/32"`, etc. | The backbone for the text encoder (usually tied to the selected CLIP image encoder). |
| **`model.prompt_learner.name`** | String | `"rankformer_prompt_learner"`, <br>`"ordinalclip_rank_prompt_learner"`, <br>`"DR_prompt_learner"` | The prompt learning strategy. Use `ordinalclip_rank_prompt_learner` to disable RankFormer and simulate the baseline. |
| **`model.prompt_learner.num_ranks`** | Integer | `3`, `4`, `5` | The number of ordinal classes/grades (e.g., 5 for Diabetic Retinopathy). |
| **`model.prompt_learner.num_tokens_for_context`**| Integer | `4`, `8`, `16`, `32` | Number of learnable context tokens prepended to the prompt. |
| **`model.prompt_learner.attn_heads`** | Integer | `1`, `2`, `4`, `8` | Number of attention heads used if using the RankFormer prompt learner. |
| **`model.sms.enabled`** | Boolean | `true`, `false` | Toggles the Similarity Matrix Smoothing (SMS) calibration module. |
| **`data.modality`** | String | `"CFP"`, `"OCT"`, `"UWF"`, `"ALL"` | The imaging modality to load. |
| **`data.k_shot`** | Integer | `1`, `2`, `4`, `8`, `16`, `32`, `100` | The number of examples per class to use in the support (training) set. |
| **`data.seed`** | Integer | `1`, `42`, `1234`, etc. | Random seed for reproducibility (especially for few-shot sampling). |
| **`data.batch_query`** | Integer | `8`, `16`, `32`, `64`, `128` | Batch size used for validation and testing sets. |
| **`data.image_size`** | Integer | `224`, `336`, `384` | Input resolution to resize images to before passing them into the model. |
| **`data.manifest`** | String | `"data/MMRDR/CFP_manifest.csv"`, etc. | Path to the CSV manifest containing image paths and labels. |
| **`data.splits_dir`** | String | `"data/MMRDR/splits"`, etc. | Directory where train/val/test split indices are stored. |
| **`data.num_workers`** | Integer | `0`, `2`, `4`, `8` | Number of CPU workers for the PyTorch DataLoader (use `0` on Windows if facing multiprocessing crashes). |
| **`loss.lambda_rank`** | Float | `0.0`, `0.5`, `1.0`, `2.0` | Weight for the ordinal ranking loss component. Set to `0.0` to disable. |
| **`loss.mu_pairwise`** | Float | `0.0`, `0.1`, `0.5`, `1.0` | Weight for the pairwise margin loss component. Set to `0.0` to disable. |
| **`loss.tau`** | Float | `0.07`, `0.1`, `1.0` | Temperature scaling parameter used in the contrastive loss function. |
| **`train.epochs`** | Integer | `50`, `100`, `200` | Maximum number of training epochs (typically small for few-shot learning). |
| **`train.optimizer`** | String | `"adamw"`, `"adam"`, `"sgd"` | The optimization algorithm to use. |
| **`train.lr`** | Float | `1e-5`, `5e-5`, `1e-4`, `1e-3` | The peak learning rate. |
| **`train.weight_decay`** | Float | `0.0`, `1e-4`, `1e-2` | L2 weight decay regularization factor. |
| **`train.scheduler`** | String | `"cosine"`, `"step"`, `"linear"`, `"none"` | Learning rate scheduler type. |
| **`train.warmup_epochs`** | Integer | `0`, `5`, `10` | Number of epochs to linearly warm up the learning rate. |
| **`train.early_stopping_patience`**| Integer | `10`, `20`, `50` | Number of epochs with no improvement before stopping training early. |
| **`train.early_stopping_monitor`** | String | `"val_qwk"`, `"val_acc"`, `"val_loss"` | The validation metric to monitor for early stopping. |
| **`train.early_stopping_mode`** | String | `"max"`, `"min"` | Whether the monitored metric should be maximized (`max`) or minimized (`min`). |
| **`metrics`** | List[String] | `["qwk", "mae", "macro_f1", "auc", "acc", "los"]` | Subset of evaluation metrics to calculate and log during validation/testing. |


In [ ]:
# ── Modality → config file + number of DR grades ─────────────────────────────
MODALITIES = {
    'CFP': {'config': 'configs/deepdrid_cfp.yaml', 'num_ranks': 5},
}
DATASET = 'DeepDRiD'  # used for labelling

# ── k-shot regimes: full data and the low shots ───────────────────────────────
K_SHOTS = [-1, 1, 2, 4]

# ── Seeds for statistical reliability ────────────────────────────────────────
SEEDS = [1, 2, 3]

# ── Per-run epoch budget of Ours (Full); variant E sets its own ─────────────
EPOCHS = 50

# ── Recipe changes taken from OrdinalCLIP (official) ─────────────────────────
RECIPE = {
    'E': {'__own_budget__': True, 'train.epochs': '100', 'train.warmup_epochs': '0'},
    'T': {'model.prompt_learner.init_context':
              "'diabetic retinopathy grading: the severity of the retinopathy is'",
          'model.prompt_learner.full_stop': 'true'},
    'F': {'model.image_encoder.full_finetune': 'true'},
}


def recipe(codes: str) -> dict:
    """Overrides of Ours (Full) plus the recipe changes in `codes`, e.g. 'ET'."""
    spec = {}
    for c in codes:
        spec.update(RECIPE[c])
    return spec


# ── Variants ─────────────────────────────────────────────────────────────────
# Ours (Full) is the reference (already in the shared cache, not retrained).
ABLATIONS = {
    'Ours (Full)':  {},
    'Ours +E':      recipe('E'),
    'Ours +T':      recipe('T'),
    'Ours +F':      recipe('F'),
    'Ours +E+T':    recipe('ET'),
    'Ours +E+F':    recipe('EF'),
    'Ours +T+F':    recipe('TF'),
    'Ours +E+T+F':  recipe('ETF'),
}
# Reference baselines read from the cache in 5d (never trained here)
REFERENCES = ['OrdinalCLIP (official)', 'CLIP-DR (official)']

# ── Quick count ───────────────────────────────────────────────────────────────
new_variants = [v for v in ABLATIONS if v != 'Ours (Full)']
print(f'k-shots     : {K_SHOTS}')
print(f'Seeds       : {SEEDS}')
print(f'New models  : {len(new_variants)}  -> {len(new_variants) * len(K_SHOTS) * len(SEEDS)} runs')


## 3 · Dataset Overview

Load manifests and plot class distributions per modality.

In [ ]:
def load_manifest(modality: str) -> pd.DataFrame | None:
    path = ROOT / 'data/DeepDRiD' / 'CFP_manifest.csv'
    if not path.exists():
        print(f'  [SKIP] manifest not found: {path}')
        return None
    return pd.read_csv(path)


manifests = {m: load_manifest(m) for m in MODALITIES}

# ── Plot class distributions (train vs. test, side by side) ──────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
})

SPLIT_STYLE = {'train': ('Train', '#4361EE'), 'test': ('Test', '#D97706')}
GRADE_NAMES = ['Grade 0\n(No DR)', 'Grade 1\n(Mild)', 'Grade 2\n(Moderate)',
               'Grade 3\n(Severe)', 'Grade 4\n(Proliferative)']
BAR_W = 0.38

available = {m: df for m, df in manifests.items() if df is not None}

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(6.4 * len(available), 3.9),
                             constrained_layout=True, squeeze=False)

    for ax, (mod, df) in zip(axes[0], available.items()):
        num_ranks = MODALITIES[mod]['num_ranks']
        grades = np.arange(num_ranks)
        y_max = 0
        for i, (split, (label, color)) in enumerate(SPLIT_STYLE.items()):
            counts = (df[df['split'] == split]['grade']
                      .value_counts().reindex(grades, fill_value=0))
            share = counts / counts.sum() * 100
            x = grades + (i - 0.5) * (BAR_W + 0.02)   # small gap between the pair
            bars = ax.bar(x, counts.values, width=BAR_W, color=color,
                          edgecolor='white', linewidth=0.8, label=f'{label} (n = {counts.sum():,})')
            for bar, n, pct in zip(bars, counts.values, share.values):
                ax.text(bar.get_x() + bar.get_width() / 2, n + 6, f'{n:,}\n{pct:.0f}%',
                        ha='center', va='bottom', fontsize=7, color='#333333', linespacing=1.1)
            y_max = max(y_max, counts.max())

        ax.set_title(f'DeepDRiD-{mod}: images per DR grade', fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel('DR grade (per eye)', fontsize=10)
        ax.set_ylabel('Number of images', fontsize=10)
        ax.set_xticks(grades)
        ax.set_xticklabels(GRADE_NAMES[:num_ranks], fontsize=8.5)
        ax.set_ylim(0, y_max * 1.18)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
        ax.tick_params(axis='both', labelsize=8.5)
        ax.legend(fontsize=9, frameon=False, loc='upper right')

    plt.savefig(FIGURES / 'dataset_distribution.pdf', bbox_inches='tight', dpi=300)
    plt.savefig(FIGURES / 'dataset_distribution.png', bbox_inches='tight', dpi=300)
    plt.show()
    print(f'Saved → {FIGURES / "dataset_distribution.pdf"}')
else:
    print('No manifests found. Run scripts/prepare_external_datasets.py first.')

## 4 · Prepare Few-Shot Splits

Generates support/query text files for each modality × k × seed combination.

In [ ]:
SPLITS_DIR = ROOT / 'data/DeepDRiD' / 'splits'

def splits_exist(modality: str, k: int, seed: int) -> bool:
    mod_lower = modality.lower()
    sup = SPLITS_DIR / f'{mod_lower}_{k}shot_s{seed}_support.txt'
    qry = SPLITS_DIR / f'{mod_lower}_{k}shot_s{seed}_query.txt'
    return sup.exists() and qry.exists()

to_generate = [
    m for m in MODALITIES
    if manifests.get(m) is not None
    and not all(splits_exist(m, k, s) for k in fs_k for s in SEEDS)
]

if to_generate:
    print('Generating splits for:', to_generate)
    kshots_str = ','.join(str(k) for k in fs_k)
    seeds_str  = ','.join(str(s) for s in SEEDS)
    for mod in to_generate:
        cmd = [
            sys.executable, 'src/data/build_splits.py',
            '--modality', mod,
            '--dataset', 'DeepDRiD',
            '--kshots', kshots_str,
            '--seeds', seeds_str,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print('ERROR:', result.stderr)
else:
    print('All splits already exist — skipping generation.')

## 5 · Training

### 5a · Single-run helper

This function trains (or skips if already done) one experiment and returns the test metrics.

In [ ]:
from src.utils.experiment import run_experiment as _run_experiment


def split_spec(extra_overrides, default_config):
    """(config file, overrides) of a variant: its own config file or overrides on the main config."""
    spec = dict(extra_overrides or {})
    config = spec.pop('__config__', default_config)
    own_budget = spec.pop('__own_budget__', False)
    overrides = ([] if own_budget else [f'train.epochs={EPOCHS}']) + [f'{k}={v}' for k, v in spec.items()]
    return config, overrides


def run_experiment(config_path: str, modality: str, k_shot: int, seed: int,
                   variant_name: str = 'Ours (Full)', extra_overrides: dict | None = None,
                   force_retrain: bool = False) -> dict:
    """Train one run, or load it from results_shared/ if any notebook already trained this config.

    Validation every epoch, best.ckpt and last.ckpt after every epoch, resume from
    last.ckpt, test with the best checkpoint (src/utils/experiment.py).
    """
    config, overrides = split_spec(extra_overrides, config_path)
    return _run_experiment(variant_name, config, k_shot, seed, SHARED_RESULTS,
                           overrides=overrides, force_retrain=force_retrain)

print('run_experiment() helper defined.')


### 5a-2 · Where Each Variant Is Saved

Resolves every variant's config without training and checks that no two variants (and no variant
and Ours (Full)) share a `results_shared/` folder. Run it before 5c.


In [ ]:
from src.utils.experiment import load_run_config, config_key

key_rows = []
for variant_name, overrides in ABLATIONS.items():
    config, overrides_list = split_spec(overrides, MODALITIES['CFP']['config'])
    cfg = load_run_config(config, 1, 1, overrides_list)
    key_rows.append({
        'Variant': variant_name,
        'folder': f'results_shared/{config_key(cfg)}',
        'epochs': cfg.train.epochs,
        'warmup': cfg.train.warmup_epochs,
        'init_context': cfg.model.prompt_learner.init_context,
        'full_stop': cfg.model.prompt_learner.full_stop,
        'full_finetune': cfg.model.image_encoder.full_finetune,
        'lora': cfg.model.image_encoder.lora.enabled,
    })
df_keys = pd.DataFrame(key_rows)
assert df_keys['folder'].is_unique, 'two variants resolve to the same config'
display(df_keys)


### 5b · Reference: Ours (Full)

Loads Ours (Full) at the same k and seeds from the shared cache (all of them are already trained).


In [ ]:
# ── Subset these to run only part of the grid ─────────────────────────────────
MODALITIES_TO_RUN = list(MODALITIES)      # e.g. ['CFP'] for a single modality
K_SHOTS_TO_RUN    = K_SHOTS
SEEDS_TO_RUN      = SEEDS

# ── Collect all results ───────────────────────────────────────────────────────
all_results = []  # list of dicts

total = len(MODALITIES_TO_RUN) * len(K_SHOTS_TO_RUN) * (
    1 if 0 in K_SHOTS_TO_RUN else 0  # zero-shot = 1 run per modality
) + len(MODALITIES_TO_RUN) * len([k for k in K_SHOTS_TO_RUN if k > 0]) * len(SEEDS_TO_RUN)

pbar = tqdm(total=total, desc='Experiments')

seen_zeroshot = set()  # zero-shot only once per modality

for mod in MODALITIES_TO_RUN:
    cfg_path = MODALITIES[mod]['config']

    for k in K_SHOTS_TO_RUN:
        seeds_for_k = [1] if k == 0 else SEEDS_TO_RUN  # zero-shot: single deterministic run

        for seed in seeds_for_k:
            if k == 0 and mod in seen_zeroshot:
                continue
            if k == 0:
                seen_zeroshot.add(mod)

            pbar.set_postfix({'mod': mod, 'k': k, 'seed': seed})

            metrics = run_experiment(
                config_path=cfg_path,
                modality=mod,
                k_shot=k,
                seed=seed,
                variant_name='Ours (Full)',
            )

            all_results.append({
                'Variant':  'Ours (Full)',
                'Modality': mod,
                'k':        k,
                'Seed':     seed,
                **metrics,
            })
            pbar.update(1)

pbar.close()

df_results = pd.DataFrame(all_results)
print('\nDone! Shape:', df_results.shape)
df_results.head(35)

### 5c · Train the Recipe Variants

Trains the 7 variants at k = -1, 1, 2, 4 for every seed. Finished runs are loaded from the shared
cache, so the cell can be stopped and re-run (an unfinished run resumes from its `last.ckpt`).


In [ ]:
RUN_ABLATIONS    = True
ABLATION_MOD     = 'CFP'
ABLATION_K_SHOTS = K_SHOTS
ABLATION_SEEDS   = SEEDS

ablation_results = []

if RUN_ABLATIONS:
    cfg_path = MODALITIES[ABLATION_MOD]['config']
    total_abl = len(ABLATIONS) * len(ABLATION_K_SHOTS) * len(ABLATION_SEEDS)
    pbar = tqdm(total=total_abl, desc='Recipe variants')

    for k_shot in ABLATION_K_SHOTS:
        for variant_name, overrides in ABLATIONS.items():
            for seed in ABLATION_SEEDS:
                pbar.set_postfix({'variant': variant_name, 'k': k_shot, 'seed': seed})
                metrics = run_experiment(
                    config_path=cfg_path,
                    modality=ABLATION_MOD,
                    k_shot=k_shot,
                    seed=seed,
                    variant_name=variant_name,
                    extra_overrides=overrides,
                )
                ablation_results.append({
                    'Variant':  variant_name,
                    'Modality': ABLATION_MOD,
                    'k':        k_shot,
                    'Seed':     seed,
                    **metrics,
                })
                pbar.update(1)
    pbar.close()

df_ablation = pd.DataFrame(ablation_results)
print('Ablation shape:', df_ablation.shape)
df_ablation.head()


### 5d · Recipe Variants vs. Ours (Full) and the Official Baselines

QWK (mean ± std over seeds) per variant and k, with OrdinalCLIP (official) and CLIP-DR (official)
read from the shared cache. Also writes `results_shared/recipe_ablation_index.csv`.


In [ ]:
frames = [df_ablation] if not df_ablation.empty else []
for name in REFERENCES:
    ref = find_runs(SHARED_RESULTS, name)
    if not ref.empty:
        frames.append(ref[ref['k'].isin(K_SHOTS)].assign(Modality='CFP'))

if frames:
    df_cmp = pd.concat(frames, ignore_index=True)
    order = list(ABLATIONS) + REFERENCES
    for m in ['qwk', 'f1', 'mae']:
        g = df_cmp.groupby(['Variant', 'k'])[m]
        t = (g.mean().round(3).astype(str) + ' ± ' + g.std().round(3).astype(str)).unstack('k')
        t = t.reindex([v for v in order if v in t.index])[[k for k in K_SHOTS if k in t.columns]]
        print(f'\n{m} (mean ± std over seeds)')
        display(t)
    df_cmp.groupby(['Variant', 'k'])[['qwk', 'f1', 'mae', 'auc']].agg(['mean', 'std']) \
          .to_csv(RESULTS / 'recipe_ablation_summary.csv')
    print(f'Saved → {RESULTS / "recipe_ablation_summary.csv"}')

# Name → folder index of the runs of this notebook
index_file = SHARED_RESULTS / 'recipe_ablation_index.csv'
df_keys.to_csv(index_file, index=False)
print(f'Saved → {index_file}')


## 6 · Aggregate & Display Results Table

In [ ]:
METRICS = ['qwk', 'mae', 'f1', 'auc', 'acc', 'los']
METRIC_LABELS = {'qwk': 'QWK↑', 'mae': 'MAE↓', 'f1': 'Macro-F1↑',
                 'auc': 'AUC↑', 'acc': 'Accuracy↑', 'los': 'LOS↑'}

def aggregate(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Compute mean ± std per group."""
    rows = []
    for keys, grp in df.groupby(group_cols):
        row = dict(zip(group_cols, keys if isinstance(keys, tuple) else [keys]))
        for m in METRICS:
            if m in grp.columns:
                row[f'{m}_mean'] = grp[m].mean()
                row[f'{m}_std']  = grp[m].std(ddof=1) if len(grp) > 1 else 0.0
                row[m] = f"{grp[m].mean():.4f} ± {row[f'{m}_std']:.4f}"
        rows.append(row)
    return pd.DataFrame(rows)

# ── Main results table ────────────────────────────────────────────────────────
if not df_results.empty:
    df_agg = aggregate(df_results, ['Variant', 'Modality', 'k'])
    display_cols = ['Variant', 'Modality', 'k'] + METRICS
    display(df_agg[display_cols].sort_values(['Modality', 'k']).to_string(index=False))

    # Save
    df_agg.to_csv(RESULTS / 'main_results.csv', index=False)
    print(f'\nSaved → {RESULTS / "main_results.csv"}')

### 7a · Confusion Matrices of Every Variant

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def get_predictions(config_path, modality, k_shot, seed, variant_name='Ours (Full)', extra_overrides=None):
    """Targets / predictions of a finished run, rebuilt from its saved confusion matrix (no retraining)."""
    config, overrides = split_spec(extra_overrides, config_path)
    m = cached_metrics(config, k_shot, seed, SHARED_RESULTS, overrides=overrides)
    if m is None or 'confusion_matrix' not in m:
        return None, None
    cm = np.asarray(m['confusion_matrix'])
    rows, cols = np.nonzero(cm)
    targets = np.repeat(rows, cm[rows, cols])
    preds = np.repeat(cols, cm[rows, cols])
    return targets, preds

# Determine what to run (falling back to all if not explicitly subsetted)
mods_to_run = locals().get('MODALITIES_TO_RUN', list(MODALITIES.keys()))
k_shots_to_run = locals().get('K_SHOTS_TO_RUN', K_SHOTS)

# Loop over all ablations, modalities, k-shots, and seeds
for variant_name, overrides in ABLATIONS.items():
    variant_slug = variant_name.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')
    
    for mod in mods_to_run:
        cfg_path = MODALITIES[mod]['config']
        num_ranks = MODALITIES[mod]['num_ranks']
        
        for k in k_shots_to_run:
            for seed in SEEDS:
                print(f"Generating CM for: {variant_name} | {mod} | k={k} | seed={seed}")
                
                targets, preds = get_predictions(cfg_path, mod, k, seed, variant_name, overrides)
                if targets is None:
                    print('  not run yet, skipped')
                    continue

                fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

                # 1. Raw count CM
                cm_counts = confusion_matrix(targets, preds)
                disp = ConfusionMatrixDisplay(cm_counts,
                                              display_labels=[f'G{i}' for i in range(num_ranks)])
                disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
                axes[0].set_title(f'Confusion Matrix (counts)\n{variant_name} - DeepDRiD-{mod}  k={k}  seed={seed}',
                                  fontsize=11, fontweight='bold')
                axes[0].grid(False) # Ensure grid is off

                # 2. Row-normalised CM (Added np.maximum to prevent div by zero just in case)
                row_sums = np.maximum(cm_counts.sum(axis=1, keepdims=True), 1)
                cm_norm = cm_counts.astype(float) / row_sums
                disp2 = ConfusionMatrixDisplay(np.round(cm_norm, 2),
                                               display_labels=[f'G{i}' for i in range(num_ranks)])
                disp2.plot(ax=axes[1], cmap='Purples', colorbar=False)
                axes[1].set_title(f'Confusion Matrix (row-normalised)\n{variant_name} - DeepDRiD-{mod}  k={k}  seed={seed}',
                                  fontsize=11, fontweight='bold')
                axes[1].grid(False) # Ensure grid is off

                # Save figure in a structured directory
                out_dir = FIGURES / variant_slug
                out_dir.mkdir(parents=True, exist_ok=True)
                save_path = out_dir / f'confusion_{mod}_k{k}_s{seed}.pdf'
                
                plt.savefig(save_path, bbox_inches='tight', dpi=300)
                plt.close(fig)
                print(f'Saved → {save_path}\n' + '-'*60)


### 7b · Each Recipe Variant vs. Ours (Full) across k

In [ ]:
# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
})

if not df_ablation.empty:

    # ── Aggregate results ─────────────────────────────────────────────────
    df_abl_agg = aggregate(
        df_ablation,
        ['Variant', 'Modality', 'k']
    )

    # ── Save/display table ─────────────────────────────────────────────────
    display_cols = (
        ['Variant', 'k'] +
        [m for m in df_abl_agg.columns if m.endswith('_mean')]
    )

    print('Ablation Study — DeepDRiD-CFP (mean over seeds)')
    print('=' * 80)

    display(
        df_abl_agg[display_cols]
        .sort_values(['Variant', 'k'])
        .head(50)
    )

    df_abl_agg.to_csv(
        RESULTS / 'ablation_results.csv',
        index=False
    )

    # ── Plot settings ─────────────────────────────────────────────────────
    PLOT_METRICS = ['qwk', 'mae', 'f1', 'auc']
    PLOT_LABELS = ['QWK ↑', 'MAE ↓', 'Macro-F1 ↑', 'AUC ↑']

    full_variant = 'Ours (Full)'
    FULL_COLOR = '#4361EE'
    ABL_COLOR = '#F72585'

    # Few-shot regimes go on the x-axis; the full-data run (k = -1) is drawn
    # as a horizontal reference line (mean ± std band) instead of a point.
    FEW_SHOT_K = [k for k in ABLATION_K_SHOTS if k >= 0]

    # All variants in the ablation results
    all_variants = (
        df_abl_agg['Variant']
        .dropna()
        .unique()
        .tolist()
    )

    # Only actual ablations, excluding Ours (Full)
    ablation_variants = [
        v for v in all_variants
        if v != full_variant
    ]

    def plot_variant(ax, data, mean_col, std_col, color, marker, label, zorder):
        """Few-shot curve with error bars + dashed full-data threshold line."""
        few = data[data['k'] >= 0].sort_values('k')
        std = few[std_col].values if std_col in few.columns else None
        ax.errorbar(
            few['k'].values,
            few[mean_col].values,
            yerr=std,
            marker=marker,
            color=color,
            linewidth=2,
            markersize=7,
            capsize=4,
            elinewidth=1.5,
            label=f'{label} (few-shot)',
            zorder=zorder
        )

        full = data[data['k'] == -1]
        if not full.empty:
            mu_full = float(full[mean_col].iloc[0])
            ax.axhline(
                mu_full,
                color=color,
                linestyle='--',
                linewidth=1.6,
                alpha=0.9,
                label=f'{label} full data ({mu_full:.3f})',
                zorder=zorder - 2
            )
            if std_col in full.columns:
                std_full = float(full[std_col].iloc[0])
                ax.axhspan(
                    mu_full - std_full,
                    mu_full + std_full,
                    color=color,
                    alpha=0.10,
                    lw=0,
                    zorder=zorder - 3
                )

    # ── Create one figure per ablation ────────────────────────────────────
    for ablation_variant in ablation_variants:

        # Get Ours (Full)
        full_data = df_abl_agg[
            (df_abl_agg['Variant'] == full_variant) &
            (df_abl_agg['Modality'] == ABLATION_MOD)
        ]

        # Get this specific ablation
        ablation_data = df_abl_agg[
            (df_abl_agg['Variant'] == ablation_variant) &
            (df_abl_agg['Modality'] == ABLATION_MOD)
        ]

        # Skip if either side is missing
        if full_data.empty or ablation_data.empty:
            print(
                f'Skipping {ablation_variant}: '
                f'missing Full or ablation results.'
            )
            continue

        # ── Four metric panels ────────────────────────────────────────────
        fig, axes = plt.subplots(
            1,
            len(PLOT_METRICS),
            figsize=(20, 4.5),
            constrained_layout=True
        )

        axes = np.atleast_1d(axes)

        for ax, metric, label in zip(
            axes,
            PLOT_METRICS,
            PLOT_LABELS
        ):

            mean_col = f'{metric}_mean'
            std_col = f'{metric}_std'

            if mean_col not in df_abl_agg.columns:
                continue

            # Ours (Full)
            plot_variant(ax, full_data, mean_col, std_col,
                         FULL_COLOR, 'o', 'Ours (Full)', zorder=10)

            # Specific ablation
            plot_variant(ax, ablation_data, mean_col, std_col,
                         ABL_COLOR, 's', ablation_variant, zorder=9)

            # ── Titles ───────────────────────────────────────────────────
            ax.set_title(
                label,
                fontsize=12,
                fontweight='bold'
            )

            ax.set_xlabel(
                'n-Shot',
                fontsize=11
            )

            # ── X-axis (few-shot regimes only) ───────────────────────────
            ax.set_xticks(FEW_SHOT_K)

            ax.set_xticklabels(
                [
                    str(k) if k > 0 else '0\n(ZS)'
                    for k in FEW_SHOT_K
                ],
                fontsize=9
            )
            ax.set_xlim(-1.5, max(FEW_SHOT_K) + 1.5)

            # ── Y-axis ───────────────────────────────────────────────────
            if metric in ('qwk', 'f1', 'auc', 'acc'):
                ax.set_ylim(0, 1.05)

            # ── Legend ───────────────────────────────────────────────────
            ax.legend(
                fontsize=8,
                framealpha=0.8,
                loc='upper right' if metric == 'mae' else 'lower right'
            )

        # ── Overall title ─────────────────────────────────────────────────
        fig.suptitle(
            f'Ablation Study: Ours (Full) vs {ablation_variant}\n'
            f'Modality: DeepDRiD-{ABLATION_MOD}',
            fontsize=14,
            fontweight='bold'
        )

        # ── Safe filename ─────────────────────────────────────────────────
        safe_variant = (
            ablation_variant
            .replace(' ', '_')
            .replace('(', '')
            .replace(')', '')
            .replace('/', '_')
            .replace('\\', '_')
            .replace(':', '')
        )

        output_path = (
            FIGURES /
            f'ablation_full_vs_{safe_variant}_kshot.pdf'
        )
        output_path_png = (
            FIGURES /
            f'ablation_full_vs_{safe_variant}_kshot.png'
        )

        plt.savefig(
            output_path,
            bbox_inches='tight',
            dpi=300
        )
        plt.savefig(
            output_path_png,
            bbox_inches='tight',
            dpi=300
        )

        plt.show()

        print(f'Saved → {output_path}')

else:
    print('No ablation results yet — run Section 5c first.')


## 9 · Quick Sanity Check (single forward pass)

Useful to confirm the model loads and runs before committing to a full sweep.

In [ ]:
import torch.nn.functional as F

SANITY_MOD = 'CFP'
SANITY_K   = 4

cfg_path = MODALITIES[SANITY_MOD]['config']
cfg = load_config(cfg_path, [f'data.k_shot={SANITY_K}', 'data.seed=1'])

model = FewShotOrdinalRunner(cfg).to(DEVICE)
model.eval()

dummy = torch.randn(4, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    img_f, txt_f = model(dummy)
    scale = model.clip_model.logit_scale.exp()
    sim   = scale * img_f @ txt_f.T
    probs = F.softmax(sim, dim=-1)
    preds = probs.argmax(dim=-1)

print(f'Image features shape : {img_f.shape}')
print(f'Text  features shape : {txt_f.shape}')
print(f'Similarity matrix    : {sim.shape}')
print(f'Probabilities        : {probs.round(decimals=4)}')
print(f'Predictions          : {preds.tolist()}')
print('\nSanity check passed!')